In [ ]:
## TO RUN ON THE CLOUD 

## preprocessed the data before starting the slide
## import embeddings <s
import os

# Define the URI to point to your manual process
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost:44123"

import fiftyone as fo

# Verify connection
print(fo.core.odm.database.get_db_conn()) 


You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information
Database(MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone'), 'fiftyone')


In [2]:
import fiftyone as fo

## list all dataset
print(fo.list_datasets())

# Close any zombie sessions that might be hanging
fo.close_app()

['FLPLAN', 'dugong', 'flplan200', 'flplanpatches']


In [3]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors


In [4]:
import sys
%load_ext autoreload
%autoreload 2

# import sys
# sys.path.append("/share/home/e2406743/code/Dugongs_IRISA-MARBEC-LIRMM/chapter2_FLPLAN/src")


In [5]:
import sys
from pathlib import Path

project_root = Path.cwd().parent 
sys.path.append(str(project_root / "src"))

from active_learning_fiftyone import compute_clustering_representativeness

# Uniqueness - Weighted Distance - Local Outlier Factor

![alt text](../presentation/images/uniqueness_explainer.svg)

## Per cluster uniqueness

![alt text](../presentation/images/per_cluster_uniqueness_all_clusters.svg)

## Ball Ranking Selection - Active Selection of Candidates

![alt text](../presentation/images/greedy_ball_ranking_aclr.svg)

#### What does it means a ball radius (r)?

Cosine similarity is related to Euclidean distance as follows. Denote Euclidean distance by the usual 
$‖A−B‖$
For any two vectors $A$ and $B$:

$$
\|A - B\|^2 = \|A\|^2 + \|B\|^2 - 2(A \cdot B) 
$$ 
(so called polarization  identity)

When both vectors are L2-normalized,

$$
\|A\|^2 = \|B\|^2 = 1,
$$

so:

$$
\|A - B\|^2
= 2 - 2(A \cdot B)
= 2(1 - \cos\theta).
$$

The cosine distance can be expressed in terms of Euclidean distance as:

$$
D_C(A,B)
=
\frac{\|A-B\|^2}{2}
\qquad
\text{when}
\qquad
\|A\|_2 = \|B\|_2 = 1.
$$

---

Rearranging to express cosine similarity in terms of the L2 distance:

$$
\cos\theta
=
1 - \frac{\|A-B\|^2}{2}.
$$

Therefore, for a ball of radius

$$
r = \texttt{ball\_radius},
$$

every point inside the ball satisfies

$$
\|A-B\| \le r.
$$

Squaring both sides:

$$
\|A-B\|^2 \le r^2.
$$

Substituting into the previous equation gives:

$$
\cos\theta
\ge
1 - \frac{r^2}{2}.
$$



| ball radius ($r$) | Cosine similarity $$(\theta)$$ | 
|-------------------|--------------------------|
| 0.10 | 0.995 | 
| 0.20 | 0.980 | 
| 0.30 | 0.955 | 
| 0.40 | 0.920 | 
| **0.50** | **0.875** | 
| 0.60 | 0.820 | 
| 0.70 | 0.755 | 


## Ball Ranking - L2 Penalized Uniqueness

![alt text](../presentation/images/soft_penalty_propagation_panels.svg)

In [6]:
dataset=fo.load_dataset("FLPLAN")

In [16]:
for sample in dataset.iter_samples(progress=True):
    sample['name_plot'] = Path(sample['filepath']).stem.split('-')[0]

   0% ||------------------|  0/83 [53.0ms elapsed, ? remaining, ? samples/s] 

 100% |███████████████████| 83/83 [1.8s elapsed, 0s remaining, 50.0 samples/s]         


In [32]:
from active_learning_fiftyone import compute_clustering

In [ ]:
compute_clustering(
    dataset,
    embeddings_field = "full_embeddings",
    cluster_field = "cluster_label_8",
    method = "kmeans",
    n_clusters = 8,
    seed = 42,
    save= True,
    verbose = True,
)

In [30]:
from plots_helper import plot_clusters_full_images

In [ ]:
plot_clusters_full_images(
    dataset,
    field_cluster = "cluster_label_8",
    num_img_per_row= 5,
    size= "a4",
    format= "jpeg",
    field_image_name = "name_plot",
    output_path = "8full_images_clusters.jpeg",
    seed  = 42,
    verbose = True,
)

# 10 clusters

![alt text](../presentation/images/full_images_clusters.jpeg)

# 8 clusters

![alt text](../presentation/images/8full_images_clusters.jpeg)

In [37]:
from active_learning_fiftyone  import compute_soft_coverage_scores

In [ ]:
compute_soft_coverage_scores(
    dataset,
    embeddings_field="full_embeddings",
    uniqueness_field="uniqueness_score_per_cluster",
    cluster_field="cluster_label_8",
    ball_radius = 0.5,
    penalty= 0.7,
    coverage_field = "soft_coverage_score",
    save = True,
    verbose = True,
)

In [42]:
from plots_helper import plot_cluster_uniqueness_overview

In [ ]:
fig = plot_cluster_uniqueness_overview(
    dataset,
    # embedding source
    embeddings_field   = "full_embeddings",
    # score fields  one per row
    global_uniqueness_field     = "uniqueness_score",
    cluster_uniqueness_field    = "uniqueness_score_per_cluster",
    soft_coverage_field         = "soft_coverage_score",
    # cluster membership 
    cluster_field   = "cluster_label_8",
    # dimensionality reduction
    umap_neighbors          = 5,
    umap_min_dist           = 0.1,
    tsne_perplexity         = 30.0,
    seed                    = 42,
    # visual
    score_cmap               = "RdYlGn",
    cluster_cmap             = "Set1",
    point_size               = 24,
    alpha                    = 0.75,
    # output
    figsize = (18, 16),
    dpi     = 150,
    save_path= "dimred_full_analysis.jpeg",
    verbose = True,
)

![alt text](../presentation/images/dimred_full_analysis.jpeg)

In [62]:
from plots_helper import plot_cluster_cosine_similarity

In [ ]:
plot_cluster_cosine_similarity(
    dataset,

    # data fields
    embeddings_field= "full_embeddings",
    cluster_field = "cluster_label_8",
    field_image_name = "name_plot",

    # layout
    n_compare = 4,
    size = "a4",
    format = "jpeg",

    # sampling & ordering
    seed   = 42,
    sort_by_similarity = True,

    # visual
    ref_border_color= (83, 74, 183),
    sim_cmap   = "RdYlGn",
    show_sim_bar = True,
    sim_bar_h  = 12,

    # output
    output_path  = "cluster_cosine_similar.jpeg",
    verbose  = True,
)

![alt text](cluster_cosine_similar.jpeg)

# Hyperparameter Optimization

Using optuna for hyperparameter search

# Seeds and Model training Evaluation

In [28]:
view_test_0 = dataset.load_saved_view('view_test0')
view_test_1= dataset.load_saved_view('view_test1')
view_test_2 = dataset.load_saved_view('view_test2')

In [19]:
from reconstruct import reconstruct_batch, reconstruct_from_json, reconstruct_and_nms

In [17]:
dataset.delete_sample_fields("novo_teste_raw")

In [10]:
import os

In [ ]:

root_folder = "/share/home/e2406743/code/Dugongs_IRISA-MARBEC-LIRMM/inference_flplan"
listdir = os.listdir(root_folder)
list_folders_jj = [os.path.join(root_folder,l) for l in listdir]
list_folders_jj

In [ ]:
# reconstruct the predictions with NMS
reconstruct_batch(
    dataset,
    json_paths= list_folders_jj,
    iou_threshold = 0.35,
    confidence_threshold = 0.1,
    run_nms = True,
    verbose  = True,
)

In [25]:
from evaluate import get_fields_for_seed, select_evaluation_list, run_evaluations, extract_all_metrics

In [24]:
# Get fields for this seed
fields_seed0 = get_fields_for_seed(dataset, seed_term="seed0")
fields_seed0['nms']

seed_term='seed0'  total=28  raw=14  nms=14  clean=28


['NWW_p40_aclr_seed0_nms',
 'NWW_p40_random_seed0_nms',
 'NWW_p50_random_seed0_nms',
 'NWW_p10_aclr_seed0_nms',
 'NWW_p5_random_seed0_nms',
 'NWW_p100_aclr_seed0_nms',
 'NWW_p5_aclr_seed0_nms',
 'NWW_p20_random_seed0_nms',
 'NWW_p30_random_seed0_nms',
 'NWW_p10_random_seed0_nms',
 'NWW_p20_aclr_seed0_nms',
 'NWW_p30_aclr_seed0_nms',
 'NWW_p50_aclr_seed0_nms',
 'NWW_p100_random_seed0_nms']

In [29]:
# reconstruct the baseline values
baseline_path_seed0 = "/share/home/e2406743/code/Dugongs_IRISA-MARBEC-LIRMM/inference_flplan/baseline_seed0_test_predictions.json"
baseline_path_seed1 = "/share/home/e2406743/code/Dugongs_IRISA-MARBEC-LIRMM/inference_flplan/baseline_seed1_test_predictions.json"
baseline_path_seed2 = "/share/home/e2406743/code/Dugongs_IRISA-MARBEC-LIRMM/inference_flplan/baseline_seed2_test_predictions.json"


In [32]:
reconstruct_and_nms(
    dataset,
    json_path= baseline_path_seed2,
    field_name = "baseline_seed2",
    iou_threshold = 0.35,
    confidence_threshold = 0.1,
    verbose = True,
)

[  ] JSON       : baseline_seed2_test_predictions.json
[  ] Field name : baseline_seed2_raw
[  ] Loading JSON ...
[  ] Loaded 32 tile entries
[  ] Grouped 32 tile entries → 9 unique sample stems  (skipped 0 unparseable)
[  ] Building dataset stem lookup ...
  47% |████████-----------| 39/83 [700.4ms elapsed, 790.2ms remaining, 55.7 samples/s] 

 100% |███████████████████| 83/83 [1.5s elapsed, 0s remaining, 58.4 samples/s]         
[  ] Dataset has 83 samples
[  ] Matched 9 / 83 samples
 100% |█████████████████████| 9/9 [451.7ms elapsed, 0s remaining, 19.9 samples/s]      
[OK  ] reconstruct_from_json done — field='baseline_seed2_raw'  total_detections=9
[  ] Running NMS (iou_threshold=0.35) → 'baseline_seed2_nms' ...
 100% |█████████████████████| 9/9 [730.4ms elapsed, 0s remaining, 12.3 samples/s]     
[OK  ] NMS done — field='baseline_seed2_nms'  before=9  after=9  suppressed=0 (0.0%)


('baseline_seed2_raw', 'baseline_seed2_nms')

In [33]:

# Build evaluation list 
sss_seed0 = select_evaluation_list(
    all_op         = fields_seed0["clean"],
    nms_or_raw     = "nms",
    seed_term      = "seed0",
    baseline_field = "baseline_seed0_nms",
)

sss_seed0


select_evaluation_list → 15 entries (nms_or_raw='nms'  seed='seed0')
  NWW_p40_aclr_seed0_nms                                  → aclr_p40_seed0_nms
  NWW_p40_random_seed0_nms                                → random_p40_seed0_nms
  NWW_p50_random_seed0_nms                                → random_p50_seed0_nms
  NWW_p10_aclr_seed0_nms                                  → aclr_p10_seed0_nms
  NWW_p5_random_seed0_nms                                 → random_p5_seed0_nms
  NWW_p100_aclr_seed0_nms                                 → aclr_p100_seed0_nms
  NWW_p5_aclr_seed0_nms                                   → aclr_p5_seed0_nms
  NWW_p20_random_seed0_nms                                → random_p20_seed0_nms
  NWW_p30_random_seed0_nms                                → random_p30_seed0_nms
  NWW_p10_random_seed0_nms                                → random_p10_seed0_nms
  NWW_p20_aclr_seed0_nms                                  → aclr_p20_seed0_nms
  NWW_p30_aclr_seed0_nms                          

[{'pred_field': 'NWW_p40_aclr_seed0_nms', 'eval_key': 'aclr_p40_seed0_nms'},
 {'pred_field': 'NWW_p40_random_seed0_nms',
  'eval_key': 'random_p40_seed0_nms'},
 {'pred_field': 'NWW_p50_random_seed0_nms',
  'eval_key': 'random_p50_seed0_nms'},
 {'pred_field': 'NWW_p10_aclr_seed0_nms', 'eval_key': 'aclr_p10_seed0_nms'},
 {'pred_field': 'NWW_p5_random_seed0_nms', 'eval_key': 'random_p5_seed0_nms'},
 {'pred_field': 'NWW_p100_aclr_seed0_nms', 'eval_key': 'aclr_p100_seed0_nms'},
 {'pred_field': 'NWW_p5_aclr_seed0_nms', 'eval_key': 'aclr_p5_seed0_nms'},
 {'pred_field': 'NWW_p20_random_seed0_nms',
  'eval_key': 'random_p20_seed0_nms'},
 {'pred_field': 'NWW_p30_random_seed0_nms',
  'eval_key': 'random_p30_seed0_nms'},
 {'pred_field': 'NWW_p10_random_seed0_nms',
  'eval_key': 'random_p10_seed0_nms'},
 {'pred_field': 'NWW_p20_aclr_seed0_nms', 'eval_key': 'aclr_p20_seed0_nms'},
 {'pred_field': 'NWW_p30_aclr_seed0_nms', 'eval_key': 'aclr_p30_seed0_nms'},
 {'pred_field': 'NWW_p50_aclr_seed0_nms', 'e

In [34]:
# Run evaluations
results_seed0 = run_evaluations(view_test_0, sss_seed0)



  [1/15]  NWW_p40_aclr_seed0_nms  →  aclr_p40_seed0_nms
Evaluating detections...
 100% |███████████████████| 14/14 [86.2ms elapsed, 0s remaining, 162.4 samples/s] 
Performing IoU sweep...
 100% |███████████████████| 14/14 [70.0ms elapsed, 0s remaining, 199.9 samples/s] 
  [2/15]  NWW_p40_random_seed0_nms  →  random_p40_seed0_nms
Evaluating detections...
 100% |███████████████████| 14/14 [45.8ms elapsed, 0s remaining, 305.6 samples/s] 
Performing IoU sweep...
 100% |███████████████████| 14/14 [50.7ms elapsed, 0s remaining, 276.3 samples/s] 
  [3/15]  NWW_p50_random_seed0_nms  →  random_p50_seed0_nms
Evaluating detections...
 100% |███████████████████| 14/14 [37.6ms elapsed, 0s remaining, 372.6 samples/s] 
Performing IoU sweep...
 100% |███████████████████| 14/14 [39.4ms elapsed, 0s remaining, 355.1 samples/s] 
  [4/15]  NWW_p10_aclr_seed0_nms  →  aclr_p10_seed0_nms
Evaluating detections...
 100% |███████████████████| 14/14 [39.1ms elapsed, 0s remaining, 358.5 samples/s] 
Performing IoU 

In [35]:
# Metrics DataFrame 
df_seed0 = extract_all_metrics(results_seed0, seed=0)

Seed 0 done — 15 / 15 models processed


In [36]:
df_seed0

,seed,eval_key,mAP,mAR,threshold,f1,precision,recall,tp,fp,fn,n_gt
0,0,aclr_p40_seed0_nms,0.0,0.0,0.01,0.0,0.0,0.0,0,32,14,14
1,0,random_p40_seed0_nms,0.0,0.0,0.01,0.0,0.0,0.0,0,37,14,14
2,0,random_p50_seed0_nms,0.0,0.0,0.01,0.0,0.0,0.0,0,21,14,14
3,0,aclr_p10_seed0_nms,0.0,0.0,0.01,0.0,0.0,0.0,0,24,14,14
4,0,random_p5_seed0_nms,0.0,0.0,0.01,0.0,0.0,0.0,0,29,14,14
5,0,aclr_p100_seed0_nms,0.0,0.0,0.01,0.0,0.0,0.0,0,25,14,14
6,0,aclr_p5_seed0_nms,0.0,0.0,0.01,0.0,0.0,0.0,0,39,14,14
7,0,random_p20_seed0_nms,0.0,0.0,0.01,0.0,0.0,0.0,0,26,14,14
8,0,random_p30_seed0_nms,0.0,0.0,0.01,0.0,0.0,0.0,0,32,14,14
9,0,random_p10_seed0_nms,0.0,0.0,0.01,0.0,0.0,0.0,0,30,14,14


In [69]:
view_test_seed0 = dataset.match_tags(
    "test_0"
)

In [71]:
view_test_seed1 = dataset.match_tags(
    "test_1"
)
view_test_seed2 = dataset.match_tags(
    "test_2"
)

In [72]:
dataset.save_view("view_test0", view_test_seed0)
dataset.save_view("view_test1", view_test_seed1)
dataset.save_view("view_test2", view_test_seed2)

In [7]:
session = fo.launch_app(dataset,
                        port=5151,
                        auto=False)

Session launched. Run `session.show()` to open the App in a cell output.


In [74]:
dataset

Name:        FLPLAN
Media type:  image
Num samples: 83
Persistent:  True
Tags:        []
Sample fields:
    id:                           fiftyone.core.fields.ObjectIdField
    filepath:                     fiftyone.core.fields.StringField
    tags:                         fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:                     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:                   fiftyone.core.fields.DateTimeField
    last_modified_at:             fiftyone.core.fields.DateTimeField
    region:                       fiftyone.core.fields.StringField
    mission_name:                 fiftyone.core.fields.StringField
    parent_name:                  fiftyone.core.fields.StringField
    ground_truth:                 fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    roi_grid:                     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels